In [1]:
import os, shutil, pathlib, json, random
import numpy as np
import cv2
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [2]:
data_root = pathlib.Path("D:/FYP/dental-vision/ML/data/dentaldataset01/YOLO")

In [3]:
CLASS_NAMES = {
    0:  "Caries",
    6:  "Missing_Teeth",
    7:  "Periapical_Lesion",
    11: "Impacted_Tooth",
    13: "Bone_Loss",
}

In [4]:
OUTPUT = pathlib.Path("D:/FYP/dental-vision/ML/data/processed_cropped")


In [5]:
("Done.")

'Done.'

In [6]:
print(f"data_root: {data_root}")

data_root: D:\FYP\dental-vision\ML\data\dentaldataset01\YOLO


In [7]:
print(f"OUTPUT: {OUTPUT}")

OUTPUT: D:\FYP\dental-vision\ML\data\processed_cropped


In [8]:
def move_bone_loss_to_test():
    # Helper to find all files in a split containing Class 13 (Bone_Loss)
    def get_files_with_class(split_name, target_class):
        label_dir = data_root / split_name / "labels"
        matching_files = []
        for lf in sorted(label_dir.glob("*.txt")):
            with open(lf, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if parts and int(parts[0]) == target_class:
                        matching_files.append(lf)
                        break
        return matching_files

    train_bone_loss_labels = get_files_with_class("train", 13)
    test_bone_loss_labels = get_files_with_class("test", 13)

    print(f"Original train Bone_Loss files: {len(train_bone_loss_labels)}")
    print(f"Original test Bone_Loss files: {len(test_bone_loss_labels)}")

    target_test_count = 150
    if len(test_bone_loss_labels) < target_test_count:
        num_to_move = target_test_count - len(test_bone_loss_labels)
        random.seed(42)
        to_move = random.sample(train_bone_loss_labels, num_to_move)
        
        test_img_dir = data_root / "test" / "images"
        test_label_dir = data_root / "test" / "labels"
        test_img_dir.mkdir(parents=True, exist_ok=True)
        test_label_dir.mkdir(parents=True, exist_ok=True)
        
        for lf in to_move:
            # Move label file
            shutil.move(str(lf), test_label_dir / lf.name)
            # Find and move corresponding image file
            img_dir = data_root / "train" / "images"
            for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
                candidate = img_dir / (lf.stem + ext)
                if candidate.exists():
                    shutil.move(str(candidate), test_img_dir / candidate.name)
                    break
        print(f"Moved {len(to_move)} Bone_Loss images and labels from train to test.")
    else:
        print("Test set already contains sufficient Bone_Loss files.")

print("Idempotent check and moving Bone_Loss from train to test in progress...")
move_bone_loss_to_test()

def convert_split(split_name):
    img_dir   = data_root / split_name / "images"
    label_dir = data_root / split_name / "labels"

    copied  = 0
    skipped = 0

    for label_file in sorted(label_dir.glob("*.txt")):
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]:
            candidate = img_dir / (label_file.stem + ext)
            if candidate.exists():
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            continue

        # Load image
        img = cv2.imread(str(img_path))
        if img is None:
            skipped += 1
            continue
            
        h, w, _ = img.shape
        
        idx = 0
        with open(label_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls_id = int(parts[0])
                if cls_id not in CLASS_NAMES:
                    continue
                
                # YOLO coordinates: class x_center y_center width height
                x_c, y_c, bw, bh = map(float, parts[1:5])
                
                # Convert to pixel coordinates
                x_c_px = x_c * w
                y_c_px = y_c * h
                w_px = bw * w
                h_px = bh * h
                
                # Add 15% padding
                pad_w = w_px * 0.15
                pad_h = h_px * 0.15
                
                xmin = int(x_c_px - (w_px / 2) - pad_w)
                xmax = int(x_c_px + (w_px / 2) + pad_w)
                ymin = int(y_c_px - (h_px / 2) - pad_h)
                ymax = int(y_c_px + (h_px / 2) + pad_h)
                
                # Clip to boundaries
                xmin = max(0, xmin)
                ymin = max(0, ymin)
                xmax = min(w, xmax)
                ymax = min(h, ymax)
                
                if xmax <= xmin or ymax <= ymin:
                    continue
                    
                crop = img[ymin:ymax, xmin:xmax]
                
                dest_folder = OUTPUT / split_name / CLASS_NAMES[cls_id]
                dest_folder.mkdir(parents=True, exist_ok=True)
                dest = dest_folder / f"{img_path.stem}_crop_{idx}.jpg"
                
                cv2.imwrite(str(dest), crop)
                copied += 1
                idx += 1

    print(f"{split_name:6s} → {copied} crops made, {skipped} skipped")

for split in ["train", "valid", "test"]:
    convert_split(split)

# Balance the train split
random.seed(42)
target_train_count = 2000
for cls_name in CLASS_NAMES.values():
    train_folder = OUTPUT / "train" / cls_name
    if train_folder.exists():
        crops = list(train_folder.glob("*.jpg"))
        if len(crops) > target_train_count:
            to_remove = random.sample(crops, len(crops) - target_train_count)
            for crop in to_remove:
                crop.unlink()
            print(f"Downsampled {cls_name} training set to {target_train_count} samples (removed {len(to_remove)}).")
        else:
            print(f"{cls_name} training set has {len(crops)} samples, no downsampling needed.")


Idempotent check and moving Bone_Loss from train to test in progress...
Original train Bone_Loss files: 1127
Original test Bone_Loss files: 150
Test set already contains sufficient Bone_Loss files.
train  → 34238 crops made, 0 skipped
valid  → 10196 crops made, 0 skipped
test   → 6194 crops made, 0 skipped
Downsampled Caries training set to 2000 samples (removed 5225).
Downsampled Missing_Teeth training set to 2000 samples (removed 538).
Downsampled Periapical_Lesion training set to 2000 samples (removed 1559).
Downsampled Impacted_Tooth training set to 2000 samples (removed 16575).
Downsampled Bone_Loss training set to 2000 samples (removed 341).


In [9]:
print(f"{'Disease':<22} {'train':>6} {'valid':>6} {'test':>6}")
print("-" * 44)

for cls_name in CLASS_NAMES.values():
    counts = []
    for split in ["train", "valid", "test"]:
        folder = OUTPUT / split / cls_name
        if folder.exists():
            counts.append(len(list(folder.glob("*.*"))))
        else:
            counts.append(0)
    print(f"{cls_name:<22} {counts[0]:>6} {counts[1]:>6} {counts[2]:>6}")

Disease                 train  valid   test
--------------------------------------------
Caries                   2000   2180   1319
Missing_Teeth            2000    541    426
Periapical_Lesion        2000   1132    600
Impacted_Tooth           2000   5859   3544
Bone_Loss                2000    484    305


In [10]:
print("Bone_Loss train/test split has been properly balanced and moved at image level.")


Bone_Loss train/test split has been properly balanced and moved at image level.


In [11]:
print(f"{'Disease':<22} {'train':>6} {'valid':>6} {'test':>6}")
print("-" * 44)

for cls_name in CLASS_NAMES.values():
    counts = []
    for split in ["train", "valid", "test"]:
        folder = OUTPUT / split / cls_name
        if folder.exists():
            counts.append(len(list(folder.glob("*.*"))))
        else:
            counts.append(0)
    print(f"{cls_name:<22} {counts[0]:>6} {counts[1]:>6} {counts[2]:>6}")

Disease                 train  valid   test
--------------------------------------------
Caries                   2000   2180   1319
Missing_Teeth            2000    541    426
Periapical_Lesion        2000   1132    600
Impacted_Tooth           2000   5859   3544
Bone_Loss                2000    484    305
